# EdgeTutor Qwen3.5-0.8B Tutor QLoRA

## Goal

Train the EdgeTutor tutoring policy on a free Colab GPU, compare the base, adapter, and merged model on the held-out split, and export a standalone 4-bit HQQ MNN model. The notebook uses the checked-in 300-row dataset; it does not call a generation API.

## Setup

Use a GPU runtime. The workflow is deterministic apart from low-level CUDA behavior. It uses the official Qwen checkpoint and current Transformers because Qwen3.5 support is recent.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers @ git+https://github.com/huggingface/transformers.git@main",
    "peft", "datasets", "accelerate", "huggingface_hub", "pandas", "safetensors", "hqq", "torchao>0.16.0", "MNN", "onnx", "yaspin", "sentencepiece", "torchvision", "pillow",
], check=True)

In [ ]:
from pathlib import Path
import subprocess
import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU before continuing."
REPO_DIR = Path("/content/edge-tutor")
MNN_DIR = Path("/content/MNN")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/attabeezy/edge-tutor.git", str(REPO_DIR)], check=True)
if not MNN_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/alibaba/MNN.git", str(MNN_DIR)], check=True)

print(torch.cuda.get_device_name(0))

## Steps

### 1. Validate and inspect the authored data

In [ ]:
import json
from collections import Counter

subprocess.run([sys.executable, str(REPO_DIR / "training/tutor/validate_dataset.py")], check=True)
train_path = REPO_DIR / "training/tutor/train.jsonl"
rows = [json.loads(line) for line in train_path.read_text(encoding="utf-8").splitlines()]
assert len(rows) == 240, f"Expected 240 training rows, found {len(rows)}"
print("train rows:", len(rows))
print("subjects:", Counter(row["subject"] for row in rows))
print("moves:", Counter(row["tutor_move"] for row in rows))

### 2. Download the upstream checkpoint

A local snapshot gives the MNN exporter a stable source directory.

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = Path("/content/qwen35-0.8b")
snapshot_download(
    repo_id="Qwen/Qwen3.5-0.8B",
    local_dir=BASE_MODEL_DIR,
    ignore_patterns=["*.gguf", "*.onnx"],
)
print(BASE_MODEL_DIR)

### 3. Train HQQ-aware QLoRA

These quantization parameters exactly match the model currently bundled by EdgeTutor: HQQ, 4-bit weights, block size 64, and a 4-bit LM head.

In [ ]:
ADAPTER_DIR = Path("/content/edgetutor-tutor-adapter")
TRAINER = MNN_DIR / "transformers/llm/finetune/mnn_qlora.py"

train_command = [
    sys.executable, str(TRAINER),
    "--base_model", str(BASE_MODEL_DIR),
    "--train_data", str(REPO_DIR / "training/tutor/train.jsonl"),
    "--validation_data", str(REPO_DIR / "training/tutor/validation.jsonl"),
    "--output_dir", str(ADAPTER_DIR),
    "--hqq",
    "--quant_bit", "4", "--quant_block", "64",
    "--lm_quant_bit", "4", "--lm_quant_block", "64",
    "--scale_bit", "16",
    "--lora_rank", "8", "--lora_alpha", "16", "--lora_dropout", "0.05",
    "--max_seq_len", "1024",
    "--per_device_train_batch_size", "1",
    "--per_device_eval_batch_size", "1",
    "--gradient_accumulation_steps", "8",
    "--learning_rate", "2e-4",
    "--warmup_steps", "5",
    "--num_train_epochs", "1",
    "--save_steps", "0",
    "--dtype", "fp16",
    "--device", "cuda:0",
    "--gradient_checkpointing",
    "--seed", "42",
]
result = subprocess.run(
    train_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result.stdout)
result.check_returncode()

### 4. Compare base and adapter

Both runs use deterministic decoding and the untouched held-out split. The CSV files include automatic policy checks and blank manual rubric columns.

In [ ]:
EVALUATOR = REPO_DIR / "training/tutor/evaluate_tutor.py"
REPORT_DIR = Path("/content/tutor-reports")
REPORT_DIR.mkdir(exist_ok=True)

for name, adapter_args in [("base", []), ("adapter", ["--adapter", str(ADAPTER_DIR)])]:
    subprocess.run([
        sys.executable, str(EVALUATOR),
        "--model", str(BASE_MODEL_DIR),
        "--input", str(REPO_DIR / "training/tutor/test.jsonl"),
        "--output", str(REPORT_DIR / f"{name}.csv"),
        "--device", "cuda:0",
        *adapter_args,
    ], check=True)

### 5. Merge the adapter and export one standalone MNN model

First create a standalone merged Hugging Face model and evaluate it. Then export that merged directory without `--lora_path`, so the Android app does not need a LoRA-loading code path.

In [ ]:
MERGED_MODEL_DIR = Path("/content/Qwen3.5-0.8B-Tutor-Merged")
MNN_MODEL_DIR = Path("/content/Qwen3.5-0.8B-Tutor-MNN")
MERGER = REPO_DIR / "training/tutor/merge_adapter.py"
EXPORTER = MNN_DIR / "transformers/llm/export/llmexport.py"

subprocess.run([
    sys.executable, str(MERGER),
    "--base", str(BASE_MODEL_DIR),
    "--adapter", str(ADAPTER_DIR),
    "--output", str(MERGED_MODEL_DIR),
], check=True)
subprocess.run([
    sys.executable, str(EVALUATOR),
    "--model", str(MERGED_MODEL_DIR),
    "--input", str(REPO_DIR / "training/tutor/test.jsonl"),
    "--output", str(REPORT_DIR / "merged.csv"),
    "--device", "cuda:0",
], check=True)

export_command = [
    sys.executable, str(EXPORTER),
    "--path", str(MERGED_MODEL_DIR),
    "--export", "mnn",
    "--hqq",
    "--quant_bit", "4", "--quant_block", "64",
    "--lm_quant_bit", "4", "--lm_quant_block", "64",
    "--scale_bit", "16",
    "--dst_path", str(MNN_MODEL_DIR),
]
export_result = subprocess.run(
    export_command,
    cwd=EXPORTER.parent,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(export_result.stdout)
export_result.check_returncode()
print(sorted(path.name for path in MNN_MODEL_DIR.iterdir()))

## Checks

Confirm the adapter improves the held-out tutoring checks before deploying. The MNN directory must contain `config.json`, `llm.mnn`, `llm.mnn.weight`, `llm_config.json`, and tokenizer files.

In [ ]:
import pandas as pd

for name in ("base", "adapter", "merged"):
    frame = pd.read_csv(REPORT_DIR / f"{name}.csv")
    automatic = [
        "route_marker_valid", "no_premature_answer", "required_content_present",
        "one_guiding_question", "within_120_words",
    ]
    print(name, frame[automatic].mean().round(3).to_dict())

required = {"config.json", "llm.mnn", "llm.mnn.weight", "llm_config.json"}
missing = required - {path.name for path in MNN_MODEL_DIR.iterdir()}
assert not missing, f"MNN export is incomplete: {sorted(missing)}"

## Next Steps

Download the reports and model archive. Replace the local `models/Qwen3.5-0.8B-MNN` directory only after confirming the adapter and merged CSVs improve the tutoring checks. Rebuild and reinstall the Android app (or clear app data) so the private model cache cannot retain the old weights.

In [ ]:
import shutil
from google.colab import drive

archive = shutil.make_archive("/content/Qwen3.5-0.8B-Tutor-MNN", "zip", MNN_MODEL_DIR)
reports_archive = shutil.make_archive("/content/edgetutor-tutor-reports", "zip", REPORT_DIR)

# files.download() cannot reliably transfer the model archive because it is
# larger than Colab's browser string limit. Save it to Drive instead.
drive.mount("/content/drive")
drive_output = Path("/content/drive/MyDrive/EdgeTutor")
drive_output.mkdir(parents=True, exist_ok=True)
model_copy = shutil.copy2(archive, drive_output / Path(archive).name)
reports_copy = shutil.copy2(reports_archive, drive_output / Path(reports_archive).name)
print("Saved model:", model_copy)
print("Saved reports:", reports_copy)